In [1]:
import pandas as pd
import sqlite3
import os

# ── LOAD ENRICHED DATA INTO SQLITE ───────────────────────────────────────────
df = pd.read_csv('zus_coffee_enriched.csv')
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')
print(f"Loaded {len(df):,} rows into SQLite\n")

os.makedirs('sql_results', exist_ok=True)

def run_query(title, sql, filename):
    print("=" * 55)
    print(f"  {title}")
    print("=" * 55)
    result = pd.read_sql_query(sql, conn)
    print(result.to_string(index=False))
    result.to_csv(f'sql_results/{filename}', index=False)
    print(f"  Saved: sql_results/{filename}\n")
    return result

Loaded 146,129 rows into SQLite



In [2]:
# ── Q1: PEAK HOURS BY ZONE ────────────────────────────────────────────────────
run_query("Q1 — Peak hours by zone", """
    SELECT
        outlet_zone,
        hour,
        COUNT(transaction_id)              AS total_orders,
        ROUND(AVG(prep_time_mins), 1)      AS avg_prep_mins,
        ROUND(AVG(hourly_order_volume), 1) AS avg_hourly_volume
    FROM orders
    GROUP BY outlet_zone, hour
    ORDER BY outlet_zone, total_orders DESC
""", "q1_peak_hours.csv")

  Q1 — Peak hours by zone
         outlet_zone  hour  total_orders  avg_prep_mins  avg_hourly_volume
KLCC / Bukit Bintang     8          6766           15.8               50.3
KLCC / Bukit Bintang    10          6727           15.6               51.7
KLCC / Bukit Bintang     9          6622           15.6               50.6
KLCC / Bukit Bintang    11          3527           10.3               26.0
KLCC / Bukit Bintang     7          3343           14.8               40.7
KLCC / Bukit Bintang    17          2774            9.2               21.3
KLCC / Bukit Bintang    14          2719            8.7               20.8
KLCC / Bukit Bintang    16          2643            8.4               21.3
KLCC / Bukit Bintang    18          2580            8.4               19.1
KLCC / Bukit Bintang    13          2580            8.3               19.6
KLCC / Bukit Bintang    15          2471            7.8               18.7
KLCC / Bukit Bintang    12          2410            7.8               18.2

,outlet_zone,hour,total_orders,avg_prep_mins,avg_hourly_volume
0,KLCC / Bukit Bintang,8,6766,15.8,50.3
1,KLCC / Bukit Bintang,10,6727,15.6,51.7
2,KLCC / Bukit Bintang,9,6622,15.6,50.6
3,KLCC / Bukit Bintang,11,3527,10.3,26.0
4,KLCC / Bukit Bintang,7,3343,14.8,40.7
5,KLCC / Bukit Bintang,17,2774,9.2,21.3
6,KLCC / Bukit Bintang,14,2719,8.7,20.8
7,KLCC / Bukit Bintang,16,2643,8.4,21.3
8,KLCC / Bukit Bintang,18,2580,8.4,19.1
9,KLCC / Bukit Bintang,13,2580,8.3,19.6


In [3]:
# ── Q2: CHANNEL BREAKDOWN BY HOUR ─────────────────────────────────────────────
run_query("Q2 — Order channel breakdown by hour", """
    SELECT
        hour,
        order_channel,
        COUNT(transaction_id)         AS total_orders,
        ROUND(AVG(prep_time_mins), 1) AS avg_prep_mins
    FROM orders
    GROUP BY hour, order_channel
    ORDER BY hour, total_orders DESC
""", "q2_channel_by_hour.csv")

  Q2 — Order channel breakdown by hour
 hour order_channel  total_orders  avg_prep_mins
    6       Walk-in          1773           12.4
    6    ShopeeFood          1626           12.9
    6      GrabFood          1086           13.2
    7       Walk-in          8602           14.6
    7    ShopeeFood          2584           14.6
    7      GrabFood          1900           14.7
    8       Walk-in         11221           15.2
    8    ShopeeFood          3441           15.4
    8      GrabFood          2567           15.1
    9       Walk-in         11301           15.3
    9    ShopeeFood          3400           15.3
    9      GrabFood          2645           15.2
   10       Walk-in         11622           15.4
   10    ShopeeFood          3693           15.3
   10      GrabFood          2702           15.3
   11    ShopeeFood          4860            9.9
   11      GrabFood          2878           10.0
   11       Walk-in          1884            9.9
   12    ShopeeFood          4

,hour,order_channel,total_orders,avg_prep_mins
0,6,Walk-in,1773,12.4
1,6,ShopeeFood,1626,12.9
2,6,GrabFood,1086,13.2
3,7,Walk-in,8602,14.6
4,7,ShopeeFood,2584,14.6
5,7,GrabFood,1900,14.7
6,8,Walk-in,11221,15.2
7,8,ShopeeFood,3441,15.4
8,8,GrabFood,2567,15.1
9,9,Walk-in,11301,15.3


In [4]:
# ── Q3: HOLIDAY IMPACT ON PREP TIME ───────────────────────────────────────────
run_query("Q3 — Holiday impact on prep time", """
    SELECT
        occasion,
        COUNT(transaction_id)              AS total_orders,
        ROUND(AVG(prep_time_mins), 1)      AS avg_prep_mins,
        ROUND(MAX(prep_time_mins), 1)      AS max_prep_mins,
        ROUND(AVG(hourly_order_volume), 1) AS avg_hourly_volume,
        ROUND(
            100.0 * SUM(CASE WHEN prep_time_mins > 20 THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                  AS pct_orders_over_20min
    FROM orders
    GROUP BY occasion
    ORDER BY avg_prep_mins DESC
""", "q3_holiday_impact.csv")

  Q3 — Holiday impact on prep time
      occasion  total_orders  avg_prep_mins  max_prep_mins  avg_hourly_volume  pct_orders_over_20min
     Raya 2026          3146           18.6           50.4               32.3                   34.8
      CNY 2026          3265           16.1           50.5               26.7                   28.0
 New Year 2026          2234           15.9           51.0               25.9                   29.2
Deepavali 2025          1133           15.7           50.3               26.0                   25.3
Christmas 2025          2048           12.9           46.0               21.0                   20.7
   Payday Week         24781           11.8           36.6               40.4                   17.7
        Normal        109522           11.8           37.3               32.3                   17.9
  Saved: sql_results/q3_holiday_impact.csv



,occasion,total_orders,avg_prep_mins,max_prep_mins,avg_hourly_volume,pct_orders_over_20min
0,Raya 2026,3146,18.6,50.4,32.3,34.8
1,CNY 2026,3265,16.1,50.5,26.7,28.0
2,New Year 2026,2234,15.9,51.0,25.9,29.2
3,Deepavali 2025,1133,15.7,50.3,26.0,25.3
4,Christmas 2025,2048,12.9,46.0,21.0,20.7
5,Payday Week,24781,11.8,36.6,40.4,17.7
6,Normal,109522,11.8,37.3,32.3,17.9


In [5]:
# ── Q4: THE CRASH POINT ────────────────────────────────────────────────────────
run_query("Q4 — Crash point: volume vs prep time", """
    SELECT
        CASE
            WHEN hourly_order_volume <= 10  THEN '01 - Low (1-10)'
            WHEN hourly_order_volume <= 20  THEN '02 - Moderate (11-20)'
            WHEN hourly_order_volume <= 30  THEN '03 - Busy (21-30)'
            WHEN hourly_order_volume <= 40  THEN '04 - Surge (31-40)'
            ELSE                                 '05 - Crash (40+)'
        END AS volume_bucket,
        COUNT(transaction_id)              AS total_orders,
        ROUND(AVG(prep_time_mins), 1)      AS avg_prep_mins,
        ROUND(MAX(prep_time_mins), 1)      AS max_prep_mins,
        ROUND(
            100.0 * SUM(CASE WHEN prep_time_mins > 20 THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                  AS pct_over_20min
    FROM orders
    GROUP BY volume_bucket
    ORDER BY volume_bucket
""", "q4_crash_point.csv")

  Q4 — Crash point: volume vs prep time
        volume_bucket  total_orders  avg_prep_mins  max_prep_mins  pct_over_20min
      01 - Low (1-10)         11099            3.8           13.3             0.0
02 - Moderate (11-20)         36831            6.9           26.2             0.6
    03 - Busy (21-30)         34797           11.0           36.4             9.5
   04 - Surge (31-40)         23805           15.2           46.0            32.5
     05 - Crash (40+)         39597           18.6           51.0            40.6
  Saved: sql_results/q4_crash_point.csv



,volume_bucket,total_orders,avg_prep_mins,max_prep_mins,pct_over_20min
0,01 - Low (1-10),11099,3.8,13.3,0.0
1,02 - Moderate (11-20),36831,6.9,26.2,0.6
2,03 - Busy (21-30),34797,11.0,36.4,9.5
3,04 - Surge (31-40),23805,15.2,46.0,32.5
4,05 - Crash (40+),39597,18.6,51.0,40.6


In [6]:
# ── Q5: ZONE COMPARISON ────────────────────────────────────────────────────────
run_query("Q5 — Zone performance comparison", """
    SELECT
        outlet_zone,
        COUNT(transaction_id)              AS total_orders,
        ROUND(AVG(prep_time_mins), 1)      AS avg_prep_mins,
        ROUND(MAX(prep_time_mins), 1)      AS max_prep_mins,
        ROUND(AVG(hourly_order_volume), 1) AS avg_hourly_vol,
        ROUND(
            100.0 * SUM(CASE WHEN prep_time_mins > 20 THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                  AS pct_over_20min,
        ROUND(
            100.0 * SUM(CASE WHEN order_channel != 'Walk-in' THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                  AS pct_online_orders
    FROM orders
    GROUP BY outlet_zone
    ORDER BY avg_prep_mins DESC
""", "q5_zone_comparison.csv")

  Q5 — Zone performance comparison
         outlet_zone  total_orders  avg_prep_mins  max_prep_mins  avg_hourly_vol  pct_over_20min  pct_online_orders
  Subang / Shah Alam         49708           12.7           51.0            34.1            20.4               57.7
KLCC / Bukit Bintang         49699           11.9           50.3            34.1            17.9               54.4
 Puchong / Cyberjaya         46722           11.8           50.4            31.5            17.8               52.1
  Saved: sql_results/q5_zone_comparison.csv



,outlet_zone,total_orders,avg_prep_mins,max_prep_mins,avg_hourly_vol,pct_over_20min,pct_online_orders
0,Subang / Shah Alam,49708,12.7,51.0,34.1,20.4,57.7
1,KLCC / Bukit Bintang,49699,11.9,50.3,34.1,17.9,54.4
2,Puchong / Cyberjaya,46722,11.8,50.4,31.5,17.8,52.1


In [7]:
# ── Q6: DRINK COMPLEXITY IMPACT ────────────────────────────────────────────────
run_query("Q6 — Drink complexity impact on prep time", """
    SELECT
        zus_category,
        drink_complexity,
        COUNT(transaction_id)              AS total_orders,
        ROUND(AVG(prep_time_mins), 1)      AS avg_prep_mins,
        ROUND(
            100.0 * SUM(CASE WHEN prep_time_mins > 20 THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                  AS pct_over_20min
    FROM orders
    GROUP BY zus_category, drink_complexity
    ORDER BY drink_complexity DESC, avg_prep_mins DESC
""", "q6_drink_complexity.csv")

  Q6 — Drink complexity impact on prep time
              zus_category  drink_complexity  total_orders  avg_prep_mins  pct_over_20min
   Matcha Strawberry Latte                 3          1210           21.3            53.4
          Buttermilk Latte                 3         45449           20.4            49.1
           Green Tea Latte                 2          6790           14.7            29.8
Roasted Hazelnut Chocolate                 2         11468           13.5            20.4
            Snack / Pastry                 1         22796            7.2             0.0
             Spanish Latte                 1         58416            6.9             0.0
  Saved: sql_results/q6_drink_complexity.csv



,zus_category,drink_complexity,total_orders,avg_prep_mins,pct_over_20min
0,Matcha Strawberry Latte,3,1210,21.3,53.4
1,Buttermilk Latte,3,45449,20.4,49.1
2,Green Tea Latte,2,6790,14.7,29.8
3,Roasted Hazelnut Chocolate,2,11468,13.5,20.4
4,Snack / Pastry,1,22796,7.2,0.0
5,Spanish Latte,1,58416,6.9,0.0


In [8]:
# ── Q7: REVENUE AT RISK ────────────────────────────────────────────────────────
run_query("Q7 — Revenue at risk (prep time > 20 mins)", """
    SELECT
        outlet_zone,
        occasion,
        COUNT(transaction_id)                                     AS total_orders,
        SUM(CASE WHEN prep_time_mins > 20 THEN 1 ELSE 0 END)     AS orders_at_risk,
        ROUND(
            100.0 * SUM(CASE WHEN prep_time_mins > 20 THEN 1 ELSE 0 END)
            / COUNT(*), 1
        )                                                         AS pct_at_risk,
        ROUND(
            SUM(CASE WHEN prep_time_mins > 20
                THEN transaction_qty * unit_price ELSE 0 END), 2
        )                                                         AS revenue_at_risk_myr
    FROM orders
    WHERE occasion != 'Normal'
    GROUP BY outlet_zone, occasion
    ORDER BY revenue_at_risk_myr DESC
""", "q7_revenue_at_risk.csv")

print("=" * 55)
print("  ALL QUERIES DONE — results saved to sql_results/")
print("=" * 55)

conn.close()


  Q7 — Revenue at risk (prep time > 20 mins)
         outlet_zone       occasion  total_orders  orders_at_risk  pct_at_risk  revenue_at_risk_myr
 Puchong / Cyberjaya    Payday Week          8030            1505         18.7              7739.85
  Subang / Shah Alam    Payday Week          8847            1501         17.0              6606.20
KLCC / Bukit Bintang    Payday Week          7904            1380         17.5              6009.75
  Subang / Shah Alam      Raya 2026          1203             478         39.7              2106.10
  Subang / Shah Alam       CNY 2026          1170             380         32.5              1651.80
 Puchong / Cyberjaya      Raya 2026           983             317         32.2              1408.45
KLCC / Bukit Bintang      Raya 2026           960             300         31.3              1301.15
  Subang / Shah Alam  New Year 2026           804             282         35.1              1290.70
 Puchong / Cyberjaya       CNY 2026          1052      

In [10]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

In [11]:
os.makedirs('charts', exist_ok=True)

df = pd.read_csv('zus_coffee_enriched.csv')
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

CRASH_THRESHOLD = 30   # orders/hour — our key finding from Q4

# ── STYLE ─────────────────────────────────────────────────────────────────────
COLORS = {
    'teal'   : '#1D9E75',
    'purple' : '#534AB7',
    'amber'  : '#BA7517',
    'coral'  : '#D85A30',
    'gray'   : '#888780',
    'red'    : '#A32D2D',
}
plt.rcParams.update({
    'font.family'     : 'DejaVu Sans',
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'figure.dpi'      : 130,
})

In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 1 — Hourly order volume heatmap by zone
# ═══════════════════════════════════════════════════════════════════════════════
pivot = df.groupby(['outlet_zone', 'hour'])['transaction_id'].count().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{h}:00" for h in pivot.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_title('Order volume heatmap by zone and hour', fontsize=13, fontweight='bold', pad=12)
ax.axvline(x=list(pivot.columns).index(11) - 0.5, color='white', linewidth=1.5, linestyle='--', alpha=0.7)
ax.axvline(x=list(pivot.columns).index(14) - 0.5, color='white', linewidth=1.5, linestyle='--', alpha=0.7)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=7, color='white' if val > pivot.values.max()*0.6 else 'black')
plt.colorbar(im, ax=ax, label='Order count')
plt.tight_layout()
plt.savefig('charts/01_heatmap.png', bbox_inches='tight')
plt.close()
print("Chart 1 saved: charts/01_heatmap.png")


Chart 1 saved: charts/01_heatmap.png


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 2 — Crash point: volume bucket vs avg prep time
# ═══════════════════════════════════════════════════════════════════════════════
buckets = df.copy()
buckets['volume_bucket'] = pd.cut(
    buckets['hourly_order_volume'],
    bins=[0, 10, 20, 30, 40, 999],
    labels=['Low\n(1-10)', 'Moderate\n(11-20)', 'Busy\n(21-30)', 'Surge\n(31-40)', 'Crash\n(40+)']
)
bucket_stats = buckets.groupby('volume_bucket', observed=True).agg(
    avg_prep=('prep_time_mins', 'mean'),
    pct_over_20=('prep_time_mins', lambda x: (x > 20).mean() * 100)
).reset_index()

fig, ax1 = plt.subplots(figsize=(9, 5))
bar_colors = [COLORS['teal'], COLORS['teal'], COLORS['amber'], COLORS['coral'], COLORS['red']]
bars = ax1.bar(bucket_stats['volume_bucket'], bucket_stats['avg_prep'],
               color=bar_colors, alpha=0.85, width=0.55)
ax1.axhline(20, color=COLORS['red'], linestyle='--', linewidth=1.5, label='20-min threshold')
ax1.set_ylabel('Avg prep time (mins)', fontsize=11)
ax1.set_xlabel('Hourly order volume', fontsize=11)
ax1.set_title('The crash point — how volume drives wait time', fontsize=13, fontweight='bold')
for bar, val in zip(bars, bucket_stats['avg_prep']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}m', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(bucket_stats['volume_bucket'], bucket_stats['pct_over_20'],
         color=COLORS['purple'], marker='o', linewidth=2, markersize=7, label='% over 20 min')
ax2.set_ylabel('% orders exceeding 20 min', fontsize=11, color=COLORS['purple'])
ax2.tick_params(axis='y', colors=COLORS['purple'])
ax2.spines['right'].set_visible(True)
lines = [
    mpatches.Patch(color=COLORS['red'], label='20-min threshold'),
    plt.Line2D([0], [0], color=COLORS['purple'], marker='o', label='% over 20 min'),
]
ax1.legend(handles=lines, loc='upper left', fontsize=9)
ax1.annotate('Crash point\n>30 orders/hr', xy=(3, 15.2), xytext=(3.3, 10),
             arrowprops=dict(arrowstyle='->', color=COLORS['red']),
             fontsize=9, color=COLORS['red'])
plt.tight_layout()
plt.savefig('charts/02_crash_point.png', bbox_inches='tight')
plt.close()
print("Chart 2 saved: charts/02_crash_point.png")


Chart 2 saved: charts/02_crash_point.png


In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 3 — Holiday prep time comparison
# ═══════════════════════════════════════════════════════════════════════════════
occasion_order = ['Normal', 'Payday Week', 'Christmas 2025',
                  'Deepavali 2025', 'New Year 2026', 'CNY 2026', 'Raya 2026']
occ_stats = df.groupby('occasion').agg(
    avg_prep=('prep_time_mins', 'mean'),
    pct_over_20=('prep_time_mins', lambda x: (x > 20).mean() * 100),
    total=('transaction_id', 'count')
).reindex(occasion_order).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = [COLORS['teal'] if o in ['Normal', 'Payday Week'] else COLORS['coral']
              for o in occ_stats['occasion']]
bar_colors[-1] = COLORS['red']   # Raya = red
bars = ax.barh(occ_stats['occasion'], occ_stats['avg_prep'],
               color=bar_colors, alpha=0.85, height=0.55)
ax.axvline(20, color=COLORS['red'], linestyle='--', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Avg prep time (mins)', fontsize=11)
ax.set_title('Holiday surge — Raya drives longest wait times', fontsize=13, fontweight='bold')
for bar, (_, row) in zip(bars, occ_stats.iterrows()):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['avg_prep']:.1f} min  ({row['pct_over_20']:.0f}% > 20min)",
            va='center', fontsize=9)
ax.set_xlim(0, 26)
ax.text(20.2, -0.6, '20-min\nlimit', fontsize=8, color=COLORS['red'])
plt.tight_layout()
plt.savefig('charts/03_holiday_impact.png', bbox_inches='tight')
plt.close()
print("Chart 3 saved: charts/03_holiday_impact.png")

Chart 3 saved: charts/03_holiday_impact.png


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# CHART 4 — 7-day rolling demand forecast vs actual
# ═══════════════════════════════════════════════════════════════════════════════
daily = df.groupby('transaction_date')['transaction_id'].count().reset_index()
daily.columns = ['date', 'actual_orders']
daily = daily.sort_values('date').reset_index(drop=True)
daily['forecast_7d'] = daily['actual_orders'].shift(1).rolling(7, min_periods=1).mean().round(0)
daily['error']       = daily['actual_orders'] - daily['forecast_7d']
daily['abs_error']   = daily['error'].abs()
daily['mape']        = (daily['abs_error'] / daily['actual_orders'] * 100).round(1)

# Tag holiday windows for annotation
raya_dates  = pd.date_range('2026-03-28', '2026-04-02')
cny_dates   = pd.date_range('2026-01-28', '2026-02-01')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily['date'], daily['actual_orders'],  color=COLORS['teal'],   linewidth=1.8, label='Actual orders', alpha=0.9)
ax.plot(daily['date'], daily['forecast_7d'],    color=COLORS['gray'],   linewidth=1.5, linestyle='--', label='7-day rolling forecast', alpha=0.85)
ax.fill_between(daily['date'], daily['actual_orders'], daily['forecast_7d'],
                where=daily['actual_orders'] > daily['forecast_7d'],
                alpha=0.15, color=COLORS['coral'], label='Under-forecast (demand surprise)')

# Shade holiday windows
for d in raya_dates:
    if daily['date'].min() <= d <= daily['date'].max():
        ax.axvspan(d, d + pd.Timedelta(days=1), alpha=0.12, color=COLORS['red'])
for d in cny_dates:
    if daily['date'].min() <= d <= daily['date'].max():
        ax.axvspan(d, d + pd.Timedelta(days=1), alpha=0.12, color=COLORS['amber'])

ax.set_title('Demand forecast vs actual — holiday spikes break the model', fontsize=13, fontweight='bold')
ax.set_ylabel('Daily orders', fontsize=11)
ax.set_xlabel('')
handles, labels = ax.get_legend_handles_labels()
extra = [
    mpatches.Patch(color=COLORS['red'],   alpha=0.3, label='Raya window'),
    mpatches.Patch(color=COLORS['amber'], alpha=0.3, label='CNY window'),
]
ax.legend(handles=handles + extra, fontsize=9, loc='upper left')

overall_mape = daily['mape'].median()
ax.text(0.99, 0.97, f'Median MAPE: {overall_mape:.1f}%', transform=ax.transAxes,
        ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
plt.tight_layout()
plt.savefig('charts/04_forecast.png', bbox_inches='tight')
plt.close()
print("Chart 4 saved: charts/04_forecast.png")

# ═══════════════════════════════════════════════════════════════════════════════
# CHART 5 — Channel mix during surge hours
# ═══════════════════════════════════════════════════════════════════════════════
surge_hours = df[df['hourly_order_volume'] > CRASH_THRESHOLD]
channel_surge = surge_hours.groupby(['hour', 'order_channel'])['transaction_id'].count().unstack(fill_value=0)
channel_surge_pct = channel_surge.div(channel_surge.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(channel_surge_pct))
ch_colors = {'ShopeeFood': COLORS['coral'], 'GrabFood': COLORS['amber'], 'Walk-in': COLORS['teal']}
for channel in ['ShopeeFood', 'GrabFood', 'Walk-in']:
    if channel in channel_surge_pct.columns:
        vals = channel_surge_pct[channel].values
        ax.bar(channel_surge_pct.index, vals, bottom=bottom,
               label=channel, color=ch_colors[channel], alpha=0.85, width=0.6)
        bottom += vals

ax.axhline(70, color=COLORS['purple'], linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(channel_surge_pct.index[-1] + 0.3, 71, '70% online benchmark', fontsize=8, color=COLORS['purple'])
ax.set_xlabel('Hour of day', fontsize=11)
ax.set_ylabel('% of orders', fontsize=11)
ax.set_title('Channel mix during surge hours (>30 orders/hr) — online dominates lunch peaks',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xticks(channel_surge_pct.index)
ax.set_xticklabels([f"{h}:00" for h in channel_surge_pct.index], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('charts/05_channel_surge.png', bbox_inches='tight')
plt.close()
print("Chart 5 saved: charts/05_channel_surge.png")


Chart 4 saved: charts/04_forecast.png
Chart 5 saved: charts/05_channel_surge.png


In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# FORECAST ACCURACY SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
holiday_mape = daily[daily['date'].isin(raya_dates.union(cny_dates))]['mape'].mean()
normal_mape  = daily[~daily['date'].isin(raya_dates.union(cny_dates))]['mape'].mean()

print("\n" + "="*55)
print("  FORECAST ACCURACY SUMMARY")
print("="*55)
print(f"Normal days  MAPE  : {normal_mape:.1f}%")
print(f"Holiday days MAPE  : {holiday_mape:.1f}%")
print(f"Forecast gap during holidays: {holiday_mape - normal_mape:.1f} percentage points worse")



  FORECAST ACCURACY SUMMARY
Normal days  MAPE  : 8.1%
Holiday days MAPE  : 14.1%
Forecast gap during holidays: 6.0 percentage points worse


In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# REVENUE AT RISK SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
at_risk = df[df['prep_time_mins'] > 20]
total_rev_at_risk = (at_risk['transaction_qty'] * at_risk['unit_price']).sum()
raya_rev_at_risk  = (at_risk[at_risk['occasion'] == 'Raya 2026']['transaction_qty'] *
                     at_risk[at_risk['occasion'] == 'Raya 2026']['unit_price']).sum()

print(f"\nTotal revenue at risk (prep > 20 min) : MYR {total_rev_at_risk:,.2f}")
print(f"Raya 2026 revenue at risk             : MYR {raya_rev_at_risk:,.2f}")
print(f"\nAll 5 charts saved to charts/ folder")



Total revenue at risk (prep > 20 min) : MYR 120,286.30
Raya 2026 revenue at risk             : MYR 4,815.70

All 5 charts saved to charts/ folder
